# Fine-tune Whisper-small for Noise-Robust Romanian Speech

Fine-tunes Whisper on Romanian podcast audio with on-the-fly MUSAN
noise augmentation. The result is the audio model used in the AVSR
shallow fusion experiments.

Pipeline:
1. Extract audio waveforms from the face-cropped AVI files
2. Download and prepare the MUSAN noise corpus
3. Load Whisper-small (already fine-tuned on Romanian by Diaconu et al.)
4. Wrap it in a dataset that adds random MUSAN noise to each clip on the fly
5. Train with HuggingFace `Seq2SeqTrainer`
6. (Optional) evaluate on a discrete grid of (noise_type × SNR) conditions

**Expected data layout** (same as `finetune_vsr.ipynb`):

```
<DATA_ROOT>/
├── ZWNM2sZxtRg/
│   ├── 00001.avi
│   ├── 00002.avi
│   └── ...
└── abc123def/
    ├── 00001.avi
    └── ...
```

The training CSV uses `file_path,transcript` where `file_path` is
`<youtube_id>/<clip_index>` (no extension).

In [ ]:
# Setup
!pip install -q jiwer torchaudio

# Optional: log into HuggingFace if you want to push checkpoints to the Hub
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace Hub")
else:
    print("HF_TOKEN not set; skipping login (push_to_hub will not work).")

In [ ]:
# Configuration

# Path to your processed face crops (one .avi per clip), organized as
# DATA_ROOT/<youtube_id>/<clip_index>.avi
DATA_ROOT = "/processed"

# Train CSV with columns (file_path, transcript), where file_path is
# <youtube_id>/<clip_index> (no extension)
TRAIN_CSV = "/train.csv"

# Cache for extracted audio waveforms (one .pt per clip at 16 kHz)
AUDIO_CACHE_DIR = "/audio_cache"

# Where to put MUSAN noise files
MUSAN_DIR = "/musan"

# Model + output
WHISPER_MODEL = "alexandradiaconu/whisper-small-echo-34"  # Romanian-tuned starting point
OUTPUT_DIR = "./whisper-small-ro-noisy"
HUB_MODEL_ID = None  # e.g. "username/whisper-small-vsro200" to push during training

# Training
MAX_TRAIN_SAMPLES = None  # None = use all
BATCH_SIZE = 8
ACCUMULATION_STEPS = 4    # effective batch size = 32
LEARNING_RATE = 1e-5
NUM_EPOCHS = 3

print(f"Data root:   {DATA_ROOT}")
print(f"Train CSV:   {TRAIN_CSV}")
print(f"Audio cache: {AUDIO_CACHE_DIR}")
print(f"MUSAN:       {MUSAN_DIR}")
print(f"Whisper:     {WHISPER_MODEL}")
print(f"Output:      {OUTPUT_DIR}")
print(f"Push to Hub: {HUB_MODEL_ID or '(local only)'}")

In [ ]:
# Extract 16 kHz mono audio from each .avi clip and cache it as a .pt
# tensor for fast random access during training. Skips clips already cached.

import os, subprocess, glob, tempfile
import torch
import torchaudio
import pandas as pd
from tqdm import tqdm


def extract_audio_from_avi(avi_path, target_sr=16000):
    """Extract mono 16 kHz waveform from an AVI as a torch.Tensor [T]."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", avi_path,
            "-vn", "-acodec", "pcm_s16le",
            "-ac", "1", "-ar", str(target_sr),
            tmp_path,
        ]
        subprocess.run(cmd, check=True)
        waveform, sr = torchaudio.load(tmp_path)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)
        if waveform.size(0) > 1:
            waveform = waveform.mean(dim=0)
        else:
            waveform = waveform.squeeze(0)
        return waveform
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


os.makedirs(AUDIO_CACHE_DIR, exist_ok=True)

df = pd.read_csv(TRAIN_CSV).dropna(subset=['transcript'])
print(f"Train CSV has {len(df)} clips")

n_extracted = 0
n_skipped = 0
n_missing = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting audio"):
    file_path = str(row['file_path'])
    avi_path = os.path.join(DATA_ROOT, file_path + ".avi")
    cache_path = os.path.join(AUDIO_CACHE_DIR, file_path + ".pt")

    if not os.path.isfile(avi_path):
        n_missing += 1
        continue

    if os.path.isfile(cache_path):
        n_skipped += 1
        continue

    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    try:
        waveform = extract_audio_from_avi(avi_path)
        torch.save(waveform, cache_path)
        n_extracted += 1
    except Exception as e:
        print(f"  Failed on {file_path}: {e}")

print(f"\nExtracted: {n_extracted}")
print(f"Skipped (already cached): {n_skipped}")
print(f"Missing source AVI: {n_missing}")

In [ ]:
# Download and extract the MUSAN noise corpus (~11 GB)
# Source: https://www.openslr.org/17/

MUSAN_URL = "https://www.openslr.org/resources/17/musan.tar.gz"
MUSAN_TAR = "/musan.tar.gz"

if os.path.exists(MUSAN_DIR) and len(os.listdir(MUSAN_DIR)) > 0:
    print("MUSAN already present")
else:
    print("Downloading MUSAN (~11 GB) ...")
    !wget -q -O {MUSAN_TAR} {MUSAN_URL}
    print("Extracting ...")
    !mkdir -p {MUSAN_DIR}
    !tar -xzf {MUSAN_TAR} -C {MUSAN_DIR} --strip-components=1
    !rm {MUSAN_TAR}
    print("MUSAN ready")

# Sanity check
for sub in ["noise", "speech", "music"]:
    n = len(glob.glob(os.path.join(MUSAN_DIR, sub, "**/*.wav"), recursive=True))
    print(f"  {sub}: {n} wav files")

In [ ]:
# MUSAN noise loader — adds random MUSAN noise on the fly during training

import random


class MUSANNoiseLoader:
    """Mixes random MUSAN noise into a waveform at a random SNR."""

    def __init__(self, musan_dir, subdirs=("noise", "speech", "music"),
                 snr_range=(-5, 10), noise_prob=0.8):
        self.snr_range = snr_range
        self.noise_prob = noise_prob
        self.noise_files = []
        for subdir in subdirs:
            d = os.path.join(musan_dir, subdir)
            if os.path.exists(d):
                self.noise_files.extend(
                    glob.glob(os.path.join(d, "**/*.wav"), recursive=True)
                )
        print(f"MUSAN loaded: {len(self.noise_files)} noise files available")

    def add_noise(self, waveform):
        """Random noise injection (used during training)."""
        if random.random() > self.noise_prob:
            return waveform

        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)

        snr_db = random.uniform(*self.snr_range)
        noise_path = random.choice(self.noise_files)
        noise, noise_sr = torchaudio.load(noise_path)

        if noise_sr != 16000:
            noise = torchaudio.functional.resample(noise, noise_sr, 16000)
        if noise.size(0) > 1:
            noise = noise.mean(dim=0, keepdim=True)

        if noise.size(1) < waveform.size(1):
            repeats = (waveform.size(1) // noise.size(1)) + 1
            noise = noise.repeat(1, repeats)
        noise = noise[:, :waveform.size(1)]

        signal_power = waveform.pow(2).mean()
        noise_power = noise.pow(2).mean()
        if noise_power > 0:
            scale = torch.sqrt(signal_power / (10 ** (snr_db / 10) * noise_power))
            waveform = waveform + scale * noise

        return waveform.squeeze(0)

    def add_noise_deterministic(self, waveform, noise_type, snr_db, rng):
        """Deterministic noise injection (used during evaluation, with a fixed RNG)."""
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)

        specific_files = [f for f in self.noise_files if noise_type in f]
        if specific_files:
            idx = torch.randint(len(specific_files), (1,), generator=rng).item()
            noise_path = specific_files[idx]
        else:
            idx = torch.randint(len(self.noise_files), (1,), generator=rng).item()
            noise_path = self.noise_files[idx]

        noise, noise_sr = torchaudio.load(noise_path)
        if noise_sr != 16000:
            noise = torchaudio.functional.resample(noise, noise_sr, 16000)
        if noise.size(0) > 1:
            noise = noise.mean(dim=0, keepdim=True)

        if noise.size(1) < waveform.size(1):
            repeats = (waveform.size(1) // noise.size(1)) + 1
            noise = noise.repeat(1, repeats)
        noise = noise[:, :waveform.size(1)]

        signal_power = waveform.pow(2).mean()
        noise_power = noise.pow(2).mean()
        if noise_power > 0:
            scale = torch.sqrt(signal_power / (10 ** (snr_db / 10) * noise_power))
            waveform = waveform + scale * noise

        return waveform.squeeze(0)

In [ ]:
# Dataset and collator

from dataclasses import dataclass
from typing import Any, Dict, List, Union
from torch.utils.data import Dataset


class WhisperPodcastDataset(Dataset):
    """Loads cached audio waveforms (.pt) and adds noise on the fly."""

    def __init__(self, csv_path, audio_cache_dir, processor,
                 noise_loader=None, max_samples=None):
        self.processor = processor
        self.noise_loader = noise_loader
        self.audio_cache_dir = audio_cache_dir

        df = pd.read_csv(csv_path).dropna(subset=['transcript'])
        raw_data = df.to_dict('records')

        self.data = []
        for item in raw_data:
            file_path = str(item['file_path'])
            a_path = os.path.join(audio_cache_dir, file_path + ".pt")
            if os.path.exists(a_path):
                self.data.append({
                    'a_path': a_path,
                    'transcript': str(item['transcript']).strip(),
                })

        if max_samples and len(self.data) > max_samples:
            self.data = random.sample(self.data, max_samples)
        print(f"Dataset ready: {len(self.data)} clips found in audio cache")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        waveform = torch.load(item['a_path'], map_location="cpu", weights_only=True).clone()
        if waveform.dim() > 1:
            waveform = waveform.squeeze(0)

        if self.noise_loader:
            waveform = self.noise_loader.add_noise(waveform)

        input_features = self.processor.feature_extractor(
            waveform.numpy(), sampling_rate=16000, return_tensors="pt"
        ).input_features.squeeze(0)

        labels = self.processor.tokenizer(item['transcript']).input_ids
        return {"input_features": input_features, "labels": labels}


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
                 ) -> Dict[str, torch.Tensor]:
        # Pad mel-spectrograms
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels and replace pad tokens with -100 so the loss ignores them
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Drop the leading BOS token if present (Trainer adds it back)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

## Fine-tuning

In [ ]:
# Initialize Whisper, processor, noise loader, dataset, and collator

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

processor = WhisperProcessor.from_pretrained(
    WHISPER_MODEL, language="Romanian", task="transcribe"
)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)

# Force Romanian transcription at generation time
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ro", task="transcribe"
)
model.generation_config.suppress_tokens = []

# Clear legacy fields from model.config so save_pretrained doesn't complain
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None

noise_loader = MUSANNoiseLoader(MUSAN_DIR)
train_dataset = WhisperPodcastDataset(
    TRAIN_CSV, AUDIO_CACHE_DIR, processor,
    noise_loader=noise_loader,
    max_samples=MAX_TRAIN_SAMPLES,
)
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
# Training arguments

training_args_kwargs = dict(
    output_dir=OUTPUT_DIR,
    # --- Memory optimizations (tuned for ~12 GB VRAM) ---
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    # --- Optimization ---
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    num_train_epochs=NUM_EPOCHS,
    # --- Logging / checkpointing ---
    eval_strategy="no",
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    # --- Misc ---
    remove_unused_columns=False,
    dataloader_num_workers=4,
)

# Optional: push to HF Hub during training
if HUB_MODEL_ID:
    training_args_kwargs.update(
        push_to_hub=True,
        hub_model_id=HUB_MODEL_ID,
        hub_strategy="checkpoint",
    )

training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

In [ ]:
# Train

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

print("Starting fine-tuning ...")
trainer.train()

# Save the final model + processor locally
FINAL_DIR = "./whisper_final_noisy"
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model saved to {FINAL_DIR}")